In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load Dataset (Assuming pima-indians-diabetes.csv is in the same directory)

In [ ]:
%pwd

In [ ]:
# 1. Load Dataset (Assuming pima-indians-diabetes.csv is in the same directory)
# Columns: Pregnancies, Glucose, BP, Skin, Insulin, BMI, Pedigree, Age, Outcome
# url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv("diabetes-pytorch/pima-indians-diabetes.data.csv", names=names)

In [ ]:
X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values

In [ ]:
# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 2. Define the Model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2) # 2 Outputs for CrossEntropy (Class 0 and Class 1)
        )

    def forward(self, x):
        logits = self.net(x)
        # We output BOTH labels and probabilities to satisfy TrustyAI and Performance metrics
        probs = torch.softmax(logits, dim=1)
        labels = torch.argmax(probs, dim=1) 
        return labels, probs

model = DiabetesModel()

In [ ]:
# 3. Quick Training Loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

X_t = torch.FloatTensor(X_train)
y_t = torch.LongTensor(y_train)

for epoch in range(100):
    optimizer.zero_grad()
    outputs_labels, outputs_probs = model(X_t)
    # Loss uses raw logits or specialized layers; here we simplify
    loss = criterion(model.net(X_t), y_t) 
    loss.backward()
    optimizer.step()

In [ ]:
# 4. Export to ONNX
model.eval()
dummy_input = torch.randn(1, 8) # Match the [1, 8] shape from your demo

torch.onnx.export(
    model,
    dummy_input,
    "diabetes.onnx",
    verbose=False,
    input_names=['dense_input'],          # Match your existing demo plumbing
    output_names=['label', 'probabilities'], # 'label' provides the INT64 TrustyAI needs
    dynamic_axes={
        'dense_input': {0: 'batch_size'},
        'label': {0: 'batch_size'},
        'probabilities': {0: 'batch_size'}
    },
    opset_version=11
)

print("✅ Model saved as diabetes.onnx")